# Account Takeover (ATO) Demo (FR-031)

This notebook provides an additive, deterministic demo of account takeover detection.
It renders standardized investigator decision outputs with deterministic fallback and case-evidence exports.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path

import pandas as pd
import nest_asyncio

from notebook_config import (
    init_notebook,
    mask_pii,
    render_decision_block,
    render_reason_codes_block,
)
from banking.analytics.detect_ato import ATODetector

In [ ]:
config = init_notebook(check_env=True, check_services=False)
config

In [ ]:
detector = ATODetector()
records = detector.detect_ato_as_records()

if not records:
    print("No live ATO alerts found. Using deterministic mock fallback (seed-42 baseline).")
    fallback_events = [
        {
            "account_id": "ACC-VICTIM-8F3A",
            "novel_device_id": "DEV-UK-8891",
            "novel_ip": "185.22.41.102",
            "event_timestamp": "2026-01-15T12:14:00Z",
            "risk_score": 0.88421,
        }
    ]
    records = detector.detect_ato_as_records(events=fallback_events)

records = sorted(records, key=lambda item: (item["alert_id"], -item["risk_score"]))
active_record = records[0] if records else None
active_record

In [ ]:
if active_record:
    evidence_df = pd.DataFrame(
        {
            "metric": [
                "Alert ID",
                "Account",
                "Novel Device",
                "Novel IP",
                "Event Timestamp",
                "Risk Score",
            ],
            "value": [
                active_record["alert_id"],
                mask_pii(active_record["account_id"]),
                mask_pii(active_record["novel_device_id"]),
                active_record["novel_ip"],
                active_record["event_timestamp"],
                active_record["risk_score"],
            ],
        }
    )
    display(evidence_df)
else:
    print("No ATO alert generated in demo scenario.")

In [ ]:
if active_record:
    decision = "BLOCK & CHALLENGE" if active_record["risk_score"] >= 0.7 else "REVIEW"
    confidence = int(round(active_record["risk_score"] * 100))

    render_decision_block(
        decision=decision,
        confidence=confidence,
        action="Initiate MFA challenge and temporarily freeze high-value outbound transfers.",
        why=active_record["rationale"],
        evidence=active_record["evidence_summary"],
    )

    render_reason_codes_block(
        reason_codes=active_record["reason_codes"],
        rationale="Device and IP novelty exceeded baseline login profile for the account.",
    )

    export_dir = Path("exports/evidence/ato")
    export_dir.mkdir(parents=True, exist_ok=True)
    timestamp_utc = datetime(2026, 1, 15, 12, 0, tzinfo=timezone.utc).isoformat()

    evidence_payload = {
        "alert_id": active_record["alert_id"],
        "timestamp": timestamp_utc,
        "detector": "ATODetector",
        "decision": decision,
        "confidence": confidence,
        "risk_score": active_record["risk_score"],
        "account_id": mask_pii(active_record["account_id"]),
        "novel_device_id": mask_pii(active_record["novel_device_id"]),
        "novel_ip": active_record["novel_ip"],
        "event_timestamp": active_record["event_timestamp"],
        "reason_codes": active_record["reason_codes"],
        "rationale": active_record["rationale"],
        "evidence_summary": active_record["evidence_summary"],
    }

    evidence_path = export_dir / f"{active_record['alert_id']}_evidence.json"
    evidence_path.write_text(json.dumps(evidence_payload, indent=2), encoding="utf-8")

    summary_df = pd.DataFrame([{**active_record}])
    summary_df["account_id"] = summary_df["account_id"].apply(mask_pii)
    summary_df["novel_device_id"] = summary_df["novel_device_id"].apply(mask_pii)
    summary_path = export_dir / "ato_alerts_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"✅ Case evidence exported: {evidence_path}")
    print(f"📊 Masked summary exported: {summary_path}")